In [142]:
import pandas as pd
import datetime as dt
import yfinance as yf
import numpy as np
from pandas import DataFrame, Series
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM
import seaborn as sns
import matplotlib.pyplot as plt


In [143]:
print("Downloading btc data from yahoo")
crypto_currency = 'BTC'
against_currency = 'USD'
start = dt.datetime(2018, 1, 1)
end = dt.datetime.now()
ticker = f'{crypto_currency}-{against_currency}'
btc_raw_dataset_1d = yf.download(ticker, start, end)
btc_raw_dataset_1d.columns = btc_raw_dataset_1d.columns.droplevel(1)
btc_raw_dataset_1d=btc_raw_dataset_1d.reset_index()

print(btc_raw_dataset_1d.columns)

print(f"Newest BTC prices in dataset:\n {btc_raw_dataset_1d.tail()}")

[*********************100%***********************]  1 of 1 completed

Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')
Newest BTC prices in dataset:
 Price       Date      Adj Close          Close           High            Low  \
2579  2025-01-23  103960.171875  103960.171875  106820.328125  101257.804688   
2580  2025-01-24  104819.484375  104819.484375  107098.546875  102772.125000   
2581  2025-01-25  104714.648438  104714.648438  105243.789062  104120.375000   
2582  2025-01-26  102682.500000  102682.500000  105438.648438  102507.710938   
2583  2025-01-28  102875.421875  102875.421875  103339.625000  101363.562500   

Price           Open        Volume  
2579   103657.671875  104104515428  
2580   103965.671875   52388229265  
2581   104824.031250   23888996502  
2582   104713.210938   22543395879  
2583   102097.468750   58957860864  


In [144]:
train_dates = pd.to_datetime(btc_raw_dataset_1d['Date'])
print(train_dates)

0      2018-01-01
1      2018-01-02
2      2018-01-03
3      2018-01-04
4      2018-01-05
          ...    
2579   2025-01-23
2580   2025-01-24
2581   2025-01-25
2582   2025-01-26
2583   2025-01-28
Name: Date, Length: 2584, dtype: datetime64[ns]


In [145]:
cols = list(btc_raw_dataset_1d)[2:7]
print(cols)

['Close', 'High', 'Low', 'Open', 'Volume']


In [146]:
df_for_training = btc_raw_dataset_1d[cols].astype(float)
print(df_for_training)

Price          Close           High            Low           Open  \
0       13657.200195   14112.200195   13154.700195   14112.200195   
1       14982.099609   15444.599609   13163.599609   13625.000000   
2       15201.000000   15572.799805   14844.500000   14978.200195   
3       15599.200195   15739.700195   14522.200195   15270.700195   
4       17429.500000   17705.199219   15202.799805   15477.200195   
...              ...            ...            ...            ...   
2579   103960.171875  106820.328125  101257.804688  103657.671875   
2580   104819.484375  107098.546875  102772.125000  103965.671875   
2581   104714.648438  105243.789062  104120.375000  104824.031250   
2582   102682.500000  105438.648438  102507.710938  104713.210938   
2583   102875.421875  103339.625000  101363.562500  102097.468750   

Price        Volume  
0      1.029120e+10  
1      1.684660e+10  
2      1.687190e+10  
3      2.178320e+10  
4      2.384090e+10  
...             ...  
2579   1.041045e+

In [147]:
scaler = StandardScaler()
scaler = scaler.fit(df_for_training)
df_for_training_scaled = scaler.transform(df_for_training)
print(df_for_training_scaled)

[[-0.65434313 -0.64768121 -0.66170692 -0.63447267 -0.85109931]
 [-0.59721511 -0.59136614 -0.66131389 -0.6555187  -0.52195682]
 [-0.5877764  -0.58594764 -0.58707803 -0.59706329 -0.52068653]
 ...
 [ 3.27194085  3.20407805  3.35573217  3.2840882  -0.16836257]
 [ 3.18431714  3.21231396  3.28450993  3.27930099 -0.23592432]
 [ 3.19263569  3.12359683  3.23397937  3.16630641  1.59242287]]


In [148]:
trainX = []
trainY = []

n_future =1
n_past =14
for x in range(n_past, len(df_for_training_scaled) - n_future + 1):
    trainX.append(df_for_training_scaled[x - n_past:x, 0:df_for_training.shape[1]])
    trainY.append(df_for_training_scaled[x + n_future - 1: x + n_future, 0])

trainX,trainY = np.array(trainX), np.array(trainY)

print('trainX shape =={}.'.format(trainX.shape))
print('trainY shape =={}.'.format(trainY.shape))


trainX shape ==(2570, 14, 5).
trainY shape ==(2570, 1).


In [149]:
model = Sequential()
# model.add(LSTM(units=50, return_sequences=True, input_shape=(trainX.shape[1], trainX.shape[2])))
# model.add(Dropout(0.2))
# model.add(LSTM(units=50, return_sequences=True))
# model.add(Dropout(0.2))
# model.add(LSTM(units=50))
# model.add(Dropout(0.2))
# model.add(Dense(trainY.shape[1]))

# model.compile(optimizer='adam', loss='mean_squared_error')

# model.fit(trainX, trainY, epochs=25, batch_size=32)
model.add(LSTM(units=64,activation='relu', input_shape=(trainX.shape[1], trainX.shape[2]), return_sequences=True))
model.add(LSTM(units=64,activation='relu', return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(trainY.shape[1]))
model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

history = model.fit(trainX,trainY,epochs=10,batch_size = 16, validation_split=0.1,verbose =1)


C:\Users\v-gawelbr\AppData\Local\anaconda3\envs\btc\lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_18 (LSTM)                       │ (None, 14, 64)              │          17,920 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_19 (LSTM)                       │ (None, 64)                  │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_11 (Dropout)                 │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 51,009 (199.25 KB)

 Trainable params: 51,009 (199.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.1485 - val_loss: 0.1466
Epoch 2/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0227 - val_loss: 0.0833
Epoch 3/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0174 - val_loss: 0.0781
Epoch 4/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0167 - val_loss: 0.1247
Epoch 5/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0154 - val_loss: 0.0201
Epoch 6/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0143 - val_loss: 0.0219
Epoch 7/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0148 - val_loss: 0.0449
Epoch 8/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0140 - val_loss: 0.0161
Epoch 9/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0132 - val_loss: 0.0785
Epoch 10/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0115 - val_loss: 0.0393


In [156]:
n_future = 90


forecast_period_dates = pd.date_range(list(train_dates)[-1], periods=n_future, freq='1d').tolist()
# print(f'Forecast period dates:\n {forecast_period_dates} ')
forecast = model.predict(trainX[-1:n_future])



forecast_copies = np.repeat(forecast, df_for_training.shape[1], axis=-1)
y_pred_future = scaler.inverse_transform(forecast_copies)[:, 0]

print(y_pred_future)
forecast_dates = []
for time_i in forecast_period_dates:
    forecast_dates.append(time_i.date())

df_forecast = pd.DataFrame({'Date': np.array(forecast_dates), 'Open': y_pred_future})
df_forecast['Date'] = pd.to_datetime(df_forecast['Date'])
print(f'df_forecast \n{df_forecast}')


original = btc_raw_dataset_1d[['Date', 'Open']]
original.loc[:,'Date'] = pd.to_datetime(original['Date'])
original = original.loc[original['Date'] >= '2019-5-1']


sns.lineplot(data=original, x='Date', y='Open', label='Original')
sns.lineplot(data=df_forecast, x='Date', y='Open', label='Forecast')

# Opcjonalne ustawienia
plt.xlabel('Date')
plt.ylabel('Open Price')
plt.title('Original vs Forecast BTC Prices')
plt.legend()

# Wyświetlenie wykresu
plt.show()






C:\Users\v-gawelbr\AppData\Local\anaconda3\envs\btc\lib\site-packages\keras\src\trainers\epoch_iterator.py:151: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


ValueError: math domain error